In [4]:
# ============================================================
# TVP-VAR Input Preprocessing Script (Minimal Version)
# ------------------------------------------------------------
# 목적
# 1) 원시 CSV 로드
# 2) 날짜 구간 2020-10-10 ~ 2026-01-12 절단
# 3) 변수별 변환
#    - SOLVPN, COPPER, GOLD, SILVER, DXY -> 100 * diff(log(x))
#    - UST10Y, VIX -> diff(x)
# 4) 날짜 기준 병합 후 complete-case만 유지
# 5) 선택적으로 표준화
# 6) TVP-VAR 입력용 최소 파일만 저장
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
BASE_DIR = Path("./original_data")
OUT_DIR = BASE_DIR / "../tvpvar_preprocessed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_MAP = {
    "SOLVPN": "SOLVPN_index.csv",
    "COPPER": "copper_futures.csv",
    "GOLD": "gold_futures.csv",
    "SILVER": "silver_futures.csv",
    "DXY": "dollar_index.csv",
    "UST10Y": "us_10y_bond_yield.csv",
    "VIX": "cboe_vix_index.csv",
}

LOG_RETURN_VARS = ["SOLVPN", "COPPER", "GOLD", "SILVER", "DXY"]
DIFF_VARS = ["UST10Y", "VIX"]

START_DATE = "2020-10-10"
END_DATE = "2026-01-12"

USE_STANDARDIZATION = True

FINAL_COLUMNS = [
    "Date",
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX",
]

# =========================
# Helper
# =========================
def standardize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def find_date_column(df: pd.DataFrame) -> str:
    for c in ["Date", "date", "DATE"]:
        if c in df.columns:
            return c
    raise ValueError(f"Date column not found. Available columns: {list(df.columns)}")

def find_value_column(df: pd.DataFrame) -> str:
    priority = [
        "Close", "close", "CLOSE",
        "Price", "price", "PRICE",
        "Adj Close", "AdjClose", "adj_close",
        "Last", "last", "LAST",
        "Value", "value", "VALUE",
    ]
    for c in priority:
        if c in df.columns:
            return c

    non_date_cols = [c for c in df.columns if c.lower() != "date"]
    if len(non_date_cols) == 1:
        return non_date_cols[0]

    raise ValueError(f"Value column not found. Available columns: {list(df.columns)}")

def clean_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.replace("%", "", regex=False)
         .str.strip(),
        errors="coerce"
    )

def load_single_series(path: Path, var_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = standardize_colnames(df)

    date_col = find_date_column(df)
    value_col = find_value_column(df)

    out = df[[date_col, value_col]].copy()
    out.columns = ["Date", var_name]

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out[var_name] = clean_numeric_series(out[var_name])

    out = out.dropna(subset=["Date", var_name]).copy()
    out = out.sort_values("Date").drop_duplicates(subset=["Date"]).reset_index(drop=True)

    return out

def apply_transform(df: pd.DataFrame, var_name: str) -> pd.DataFrame:
    out = df.copy()

    if var_name in LOG_RETURN_VARS:
        invalid_mask = out[var_name] <= 0
        out.loc[invalid_mask, var_name] = np.nan
        out[f"dlog_{var_name}"] = 100.0 * np.log(out[var_name]).diff()

    elif var_name in DIFF_VARS:
        out[f"d_{var_name}"] = out[var_name].diff()

    else:
        raise ValueError(f"Unknown transform rule for variable: {var_name}")

    return out

def merge_all_series() -> pd.DataFrame:
    merged = None

    for var_name, file_name in FILE_MAP.items():
        path = BASE_DIR / file_name
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path.resolve()}")

        df = load_single_series(path, var_name)
        df = apply_transform(df, var_name)

        keep_cols = ["Date"]
        if var_name in LOG_RETURN_VARS:
            keep_cols.append(f"dlog_{var_name}")
        else:
            keep_cols.append(f"d_{var_name}")

        df = df[keep_cols].copy()

        if merged is None:
            merged = df
        else:
            merged = pd.merge(merged, df, on="Date", how="outer")

    merged = merged.sort_values("Date").reset_index(drop=True)
    return merged

def apply_date_filter(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)

    out = df[(df["Date"] >= start_dt) & (df["Date"] <= end_dt)].copy()
    out = out.sort_values("Date").reset_index(drop=True)
    return out

def zscore_standardize(df: pd.DataFrame, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = ["Date"]

    out = df.copy()
    numeric_cols = [c for c in out.columns if c not in exclude_cols]

    means = out[numeric_cols].mean()
    stds = out[numeric_cols].std(ddof=0)

    zero_std_cols = stds[stds == 0].index.tolist()
    if zero_std_cols:
        raise ValueError(f"Zero-std columns found: {zero_std_cols}")

    out[numeric_cols] = (out[numeric_cols] - means) / stds
    return out

def save_metadata(final_df: pd.DataFrame, out_dir: Path):
    meta_path = out_dir / "preprocessing_metadata.txt"
    with open(meta_path, "w", encoding="utf-8") as f:
        f.write("=== TVP-VAR Preprocessing Metadata ===\n")
        f.write(f"START_DATE: {START_DATE}\n")
        f.write(f"END_DATE: {END_DATE}\n")
        f.write(f"USE_STANDARDIZATION: {USE_STANDARDIZATION}\n")
        f.write(f"ROWS: {len(final_df)}\n")
        if len(final_df) > 0:
            f.write(f"FINAL_START: {final_df['Date'].min()}\n")
            f.write(f"FINAL_END: {final_df['Date'].max()}\n")
        f.write(f"COLUMNS: {final_df.columns.tolist()}\n")

# =========================
# Main
# =========================
def main():
    print("=== TVP-VAR preprocessing started ===")

    merged = merge_all_series()
    merged = apply_date_filter(merged, START_DATE, END_DATE)

    final_df = merged.dropna().copy()
    final_df = final_df.sort_values("Date").reset_index(drop=True)
    final_df = final_df[FINAL_COLUMNS]

    if final_df.empty:
        raise ValueError("No data left after preprocessing.")

    # 변환본 저장
    final_df.to_csv(
        OUT_DIR / "tvpvar_input_transformed.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # 표준화본 저장
    if USE_STANDARDIZATION:
        scaled_df = zscore_standardize(final_df, exclude_cols=["Date"])
        scaled_df.to_csv(
            OUT_DIR / "tvpvar_input_scaled.csv",
            index=False,
            encoding="utf-8-sig"
        )

    # 메타데이터 저장
    save_metadata(final_df, OUT_DIR)

    print(final_df.head())
    print("\nRows:", len(final_df))
    print("Date range:", final_df["Date"].min(), "~", final_df["Date"].max())
    print(f"Saved to: {OUT_DIR.resolve()}")
    print("=== TVP-VAR preprocessing finished ===")

if __name__ == "__main__":
    main()

=== TVP-VAR preprocessing started ===
        Date  dlog_SOLVPN  dlog_COPPER  dlog_GOLD  dlog_SILVER  dlog_DXY  \
0 2020-10-13    -0.490200    -0.638457  -1.787962    -4.624306  0.462924   
1 2020-10-14    -1.336381     0.196883   0.680149     1.096376 -0.181914   
2 2020-10-15    -0.053992     1.140821   0.094538    -0.703432  0.531967   
3 2020-10-19    -0.504215     0.601286   0.278208     1.937839 -0.267226   
4 2020-10-20     0.125274     1.989158   0.204226     1.135324 -0.405406   

   d_UST10Y  d_VIX  
0    -0.035   1.00  
1    -0.002   0.33  
2     0.012   0.57  
3     0.023   1.77  
4     0.014   0.17  

Rows: 1012
Date range: 2020-10-13 00:00:00 ~ 2026-01-12 00:00:00
Saved to: D:\University\3-2.5\PADA_Lab\DC_and_meterials_2\02_Data_EDA\tvpvar_preprocessed
=== TVP-VAR preprocessing finished ===
